# Tutorial 7: Migrating from Legacy Format

This tutorial covers how to migrate data and models from the legacy deepSCENIC format (pre package release) to the new unified format. It's only relevant if you used to work in the legacy codebase directly.

## Setup

In [1]:
import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

deepSCENIC version: 0.1.0


## Legacy vs New Format

### Legacy Format (Multiple Files)

The legacy format stored data and models across multiple separate files:

**Data files (from preprocessing):**
- `raw_exprMat_train.h5ad` - RNA expression data
- `fragment_matrix_train.h5ad` - ATAC accessibility data
- `r2gpenalty_train.npz` - Region-to-gene distance penalty matrix
- `TFs.txt` - List of transcription factors

**Model files (from training):**
- `best_model.pth` - VAE checkpoint (encoder, decoders, E2 weights)
- `best_model_tf2r.pth` - MotifNet checkpoint
- `best_model_tf2r_encoder.pth` - Enformer checkpoint (~1GB)
- `E1.pkl` - Pre-computed TF→region matrix

### New Format (Unified)

**Data:** Single `.h5mu` file from `ds.pp.create_mudata()` + `ds.pp.compute_r2g_penalty()`

**Model:** Single `.pt` file containing:
- All model weights (VAE, MotifNet, Enformer)
- Pre-computed E1 matrix
- Feature names (genes, regions, TFs)
- Training configuration

## Step 1: Loading Legacy Data

Legacy models were trained with separate `.h5ad` and `.npz` files. To load a legacy model, you need the original training data files that were used during training.

Use `ds.tl.load_legacy_data()` to load these files into a MuData with the exact dimensions needed.

In [2]:
# Set your legacy data directory (where the original training files are)
import os

data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "dir not found"

# Load legacy training data using the original files
# These files must match exactly what was used during model training
mdata = ds.tl.load_legacy_data(
    rna_path=f"{data_dir}/raw_exprMat_train.h5ad",
    atac_path=f"{data_dir}/fragment_matrix_train.h5ad",
    r2g_path=f"{data_dir}/r2gpenalty_train.npz",
    tf_path=f"{data_dir}/TFs.txt",
)

print(f"\nLoaded legacy data:")
print(f"  Genes: {mdata['rna'].n_vars}")
print(f"  Regions: {mdata['atac'].n_vars}")
print(f"  TFs: {mdata['rna'].var['is_tf'].sum()}")
print(f"  R2G links: {mdata.uns['r2g']['matrix'].nnz}")

deepscenic.tl - INFO - Loading RNA data from ../../../../data/deepSCENIC/MM_lines//raw_exprMat_train.h5ad
deepscenic.tl - INFO -   RNA: 748 cells × 11717 genes
deepscenic.tl - INFO - Loading ATAC data from ../../../../data/deepSCENIC/MM_lines//fragment_matrix_train.h5ad
deepscenic.tl - INFO -   ATAC: 748 cells × 273545 regions
deepscenic.tl - INFO - Loading R2G matrix from ../../../../data/deepSCENIC/MM_lines//r2gpenalty_train.npz
deepscenic.tl - INFO -   R2G: 273545 regions × 11717 genes, 3204149 links
deepscenic.tl - INFO -   Loaded 1386 TFs from ../../../../data/deepSCENIC/MM_lines/TFs.txt
deepscenic.tl - INFO -   Marked 1386 TFs in RNA data
deepscenic.tl - INFO -   Parsing ATAC region coordinates...
deepscenic.tl - INFO - Creating MuData...
deepscenic.tl - INFO - Created MuData: 11717 genes, 273545 regions, 1386 TFs



Loaded legacy data:
  Genes: 11717
  Regions: 273545
  TFs: 1386
  R2G links: 3204149


## Step 2: Loading Legacy Models

Use `ds.tl.load_legacy_model()` to load a trained model from the legacy 3-file format.

**Requirements:**
- The 3 model checkpoint files (.pth)
- MuData from `ds.tl.load_legacy_data()` with exact training dimensions
- The pre-computed E1 matrix (E1.pkl) - optional but recommended

If the E1 matrix is not provided, the function will attempt to recalculate it (requires a registered genome via `ds.register_genome()`).

In [3]:
# Set your paths to the legacy model checkpoints
legacy_results_dir = "../../../../data/deepSCENIC/MM_lines_legacy_results/MM_lines"

# Load legacy model
model = ds.tl.load_legacy_model(
    # Model checkpoints
    vae_path=f"{legacy_results_dir}/best_model.pth",
    tf2rnet_path=f"{legacy_results_dir}/best_model_tf2r.pth",
    enformer_path=f"{legacy_results_dir}/best_model_tf2r_encoder.pth",
    # MuData from load_legacy_data (contains TF annotations, r2g matrix)
    mdata=mdata,
    # Pre-computed E1 matrix (optional, leave None to recalculate)
    e1_path=f"{legacy_results_dir}/E1.pkl",
    # Device
    device="cpu",  # or "cuda"
)

print(f"\nModel loaded successfully!")
print(f"  TFs: {len(model.tf_names)}")
print(f"  Genes: {len(model.gene_names)}")
print(f"  Regions: {len(model.region_names)}")
print(f"  E1 shape: {model.adj_E1.shape}")

deepscenic.tl - INFO - Extracting data from h5mu...
deepscenic.tl - INFO -   RNA: 11717 genes, ATAC: 273545 regions, TFs: 1386
deepscenic.tl - INFO -   R2G links: 3204149
deepscenic.tl - INFO - Loading VAE...
deepscenic.tl - INFO -   VAE loaded (epoch 232)
deepscenic.tl - INFO - Loading MotifNet...
deepscenic.tl - INFO -   MotifNet loaded (epoch 232)
deepscenic.tl - INFO - Loading Enformer...
deepscenic.tl - INFO -   Enformer loaded (epoch 232)
deepscenic.tl - INFO - Loading E1 from ../../../../data/deepSCENIC/MM_lines_legacy_results/MM_lines/E1.pkl...
deepscenic.tl - INFO -   E1 shape: torch.Size([281820, 1386])
deepscenic.tl - INFO - Legacy model loaded successfully!



Model loaded successfully!
  TFs: 1386
  Genes: 11717
  Regions: 273545
  E1 shape: torch.Size([281820, 1386])


## Step 3: Loading Full Data for Inference

The training data files (`*_train.h5ad`) only contain the training split. To run inference on **all cells** (including test cells), load the full data files instead.

Since inference doesn't require the R2G matrix (which is only needed for model loading), you can omit `r2g_path`:

In [5]:
# Load full data (all cells, not just training split)
mdata_full = ds.tl.load_legacy_data(
    rna_path=f"{data_dir}/raw_exprMat.h5ad",  # Full file, not *_train.h5ad
    atac_path=f"{data_dir}/fragment_matrix.h5ad",
    tf_path=f"{data_dir}/TFs.txt",
    # r2g_path not needed for inference
)

print(f"\nFull data loaded:")
print(f"  Cells: {mdata_full.n_obs}")
print(f"  Genes: {mdata_full['rna'].n_vars}")
print(f"  Regions: {mdata_full['atac'].n_vars}")
print(f"  TFs: {mdata_full['rna'].var['is_tf'].sum()}")

deepscenic.tl - INFO - Loading RNA data from ../../../../data/deepSCENIC/MM_lines//raw_exprMat.h5ad
deepscenic.tl - INFO -   RNA: 936 cells × 12188 genes
deepscenic.tl - INFO - Loading ATAC data from ../../../../data/deepSCENIC/MM_lines//fragment_matrix.h5ad
deepscenic.tl - INFO -   ATAC: 936 cells × 281820 regions
deepscenic.tl - INFO -   R2G matrix not provided (inference-only mode)
deepscenic.tl - INFO -   Loaded 1386 TFs from ../../../../data/deepSCENIC/MM_lines/TFs.txt
deepscenic.tl - INFO -   Marked 1386 TFs in RNA data
deepscenic.tl - INFO -   Parsing ATAC region coordinates...
deepscenic.tl - INFO - Creating MuData...
deepscenic.tl - INFO - Created MuData: 12188 genes, 281820 regions, 1386 TFs



Full data loaded:
  Cells: 936
  Genes: 12188
  Regions: 281820
  TFs: 1386


## Step 4: Saving in New Format

Once loaded, save your data and model in the new unified format for easier future use.

### Saving Data

In [7]:
# Save MuData to new format
mdata.write(data_dir + "converted_data.h5mu")
mdata_full.write(data_dir + "converted_data_full.h5mu")

# Verify by reloading
mdata_full_reloaded = ds.read(data_dir + "converted_data_full.h5mu")
print(f"Reloaded: {mdata_full_reloaded.n_obs} cells")

/home/luna.kuleuven.be/u0166574/Desktop/projects/deepSCENIC/src/deepscenic/_io.py:73: SchemaWarning: atac.var missing 'split' column
  validate_schema(mdata, strict=strict)
/home/luna.kuleuven.be/u0166574/Desktop/projects/deepSCENIC/src/deepscenic/_io.py:73: SchemaWarning: mdata.obs missing 'split' column
  validate_schema(mdata, strict=strict)
/home/luna.kuleuven.be/u0166574/Desktop/projects/deepSCENIC/src/deepscenic/_io.py:73: SchemaWarning: mdata.uns missing 'r2g' (required for training)
  validate_schema(mdata, strict=strict)
deepscenic.io - INFO - Loaded ../../../../data/deepSCENIC/MM_lines/converted_data_full.h5mu: 936 cells, 12188 genes, 281820 regions


Reloaded: 936 cells


### Saving Model

In [8]:
# Save model to new unified format
model.save(data_dir + "weights/converted_model.pt")

# Verify by reloading
model_reloaded = ds.tl.load_model(data_dir + "weights/converted_model.pt")
print(f"Reloaded model with {len(model_reloaded.tf_names)} TFs")

deepscenic.tl - INFO - Saved model to ../../../../data/deepSCENIC/MM_lines/weights/converted_model.pt
deepscenic.tl - INFO - Loaded model from ../../../../data/deepSCENIC/MM_lines/weights/converted_model.pt: 11717 genes, 1386 TFs, 273545 regions


Reloaded model with 1386 TFs


```{note}
The new model format includes all components in a single file:
- VAE, MotifNet, and Enformer weights
- Pre-computed E1 matrix
- Feature names and training config

Future loading only requires: `model = ds.tl.load_model("model.pt")`
```